# downloads and imports

In [ ]:
!pip install fastapi[all] uvicorn python-multipart

In [ ]:
# This downloads and sets up the Ngrok executable in the Google Colab instance
# Import the ngrok GPG key
!curl -s https://ngrok-agent.s3.amazonaws.com/ngrok.asc | gpg --import -

# Add the ngrok repository to the apt sources list
!echo "deb https://ngrok-agent.s3.amazonaws.com buster main" | sudo tee /etc/apt/sources.list.d/ngrok.list

# Fetch the public key associated with the ngrok repository
!sudo apt-key adv --keyserver keyserver.ubuntu.com --recv-keys 0E61D3BBAAEE37FE

# Update the apt package lists
!sudo apt-get update

# Install ngrok
!sudo apt-get install ngrok


gpg: key 0E61D3BBAAEE37FE: "ngrok agent apt repo release bot <release-bot@ngrok.com>" not changed
gpg: Total number processed: 1
gpg:              unchanged: 1
deb https://ngrok-agent.s3.amazonaws.com buster main
Executing: /tmp/apt-key-gpghome.LyYF0UXmUy/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys 0E61D3BBAAEE37FE
gpg: key 0E61D3BBAAEE37FE: "ngrok agent apt repo release bot <release-bot@ngrok.com>" not changed
gpg: Total number processed: 1
gpg:              unchanged: 1
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Ign:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy Release
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 https://ngrok-agent.s3.amazonaws.com buster InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-updat

In [ ]:
!pip install faster_whisper

In [ ]:
import torch

In [ ]:
!pip install faster_whisper
!pip install fastcoref
!pip install umap
!pip install hdbscan
!pip install faster_whisper
!pip install sentence_transformers
!pip install scikit-learn
!pip install fastapi[all] uvicorn python-multipart

In [ ]:
!ngrok authtoken 2l3PpTiFhpUlR2WFnFWrTwaKkdy_35z2t312rtp629Vbw8w4e

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
%%writefile app.py

# #downloads
# # !pip install faster_whisper
# # !pip install fastcoref
# # !pip install umap
# # !pip install hdbscan
# # !pip install faster_whisper
# # !pip install sentence_transformers
# # !pip install scikit-learn
# # !pip install fastapi[all] uvicorn python-multipart
# ##############################
# import torch
# import time
# #imports
import io
import os
import spacy
import fastcoref
import torch
import re
import spacy
import time
import torch
from fastcoref import spacy_component
import umap
import sklearn
import hdbscan
import time
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from faster_whisper import WhisperModel
from sentence_transformers import SentenceTransformer
import time
from typing import Any
from pydantic import BaseModel
from fastapi import FastAPI, File, UploadFile, HTTPException,Form,Body
from sklearn.decomposition import PCA
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.cluster import KMeans
from faster_whisper import WhisperModel
from sklearn.feature_extraction.text import CountVectorizer
import shutil
import os
import tempfile

nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("fastcoref")
##############################

#helper functions

def read_file(path):
  with open(path, 'r') as file:
    file_content = file.read()
  return file_content

# Function to clean the transcription text
def clean_transcription(text):
    # Remove timestamps
    text = re.sub(r'\d{2}:\d{2}:\d{2},\d{3} --> \d{2}:\d{2}:\d{2},\d{3}', '', text)
    # Remove segment numbers
    text = re.sub(r'^\d+\s*$', '', text, flags=re.MULTILINE)
    # Remove extra blank lines
    text = re.sub(r'\n\s*\n', '\n', text)
    return text.strip()

def coreference_resolution(text):
  doc = nlp(
  text,
  component_cfg={"fastcoref": {'resolve_text': True}})
  return doc._.resolved_text

def embed_sentences(text,model_name="paraphrase-MiniLM-L6-v2"):
    model = SentenceTransformer(model_name)
    doc = nlp(text)
    sentences = [sent.text for sent in doc.sents]
    embeddings = model.encode(sentences)
    return sentences, embeddings

def segment_topics_hdbscan(embeddings,min_cluster_size=3,min_samples=2,cluster_selection_epsilon=0.8):
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples, cluster_selection_epsilon=cluster_selection_epsilon)
    cluster_labels = clusterer.fit_predict(embeddings)
    return cluster_labels

def putting_sentences_in_cluster(cluster_labels, sentences):
    cluster_to_sentences = {}

    for i, label in enumerate(cluster_labels):
        if label not in cluster_to_sentences:
            cluster_to_sentences[label] = []
        cluster_to_sentences[label].append(sentences[i])

    return cluster_to_sentences

def printing_clusters(clusters):
  # Print out sentences in each cluster
  for label, sentences_in_cluster in clusters.items():
      print(f"Topic {label + 1}:")
      for sentence in sentences_in_cluster:
          print(f" - {sentence}")
      print("\n")

def cluster_as_string(clusters):
    output_string = ""  # Initialize an empty string to store the output

    # Loop through each cluster and concatenate the sentences into the string
    for label, sentences_in_cluster in clusters.items():
        output_string += f"Topic {label + 1}:\n"
        for sentence in sentences_in_cluster:
            output_string += f" - {sentence}\n"
        output_string += "\n"
        # Add a newline to separate clusters

#     return output_string

def generate_topic_title(sentences):
    vectorizer = CountVectorizer(max_features=3, stop_words='english')
    X = vectorizer.fit_transform(sentences)
    print(X,"rrdf")
    keywords = vectorizer.get_feature_names_out()
    title = ' '.join(keywords)
    return title

def get_clusters_as_string_with_titles(clusters):
    output_string = ""  # Initialize an empty string to store the output

    # Loop through each cluster and concatenate the sentences into the string
    for label, sentences_in_cluster in clusters.items():
        # Generate a title for the current cluster
        title = generate_topic_title(sentences_in_cluster)

        # Append the title and the topic number to the output string
        output_string += f"Topic {label + 1}: {title}\n"

        # Append each sentence to the output string
        for sentence in sentences_in_cluster:
            output_string += f" - {sentence}\n"
        output_string += "\n"  # Add a newline after each topic for better formatting

    return output_string

def get_clusters_with_titles(clusters):
    # Initialize an empty dictionary to store the output
    clusters_with_titles = {}

    # Loop through each cluster and generate the title
    for label, sentences_in_cluster in clusters.items():
        # Generate a title for the current cluster
        title = generate_topic_title(sentences_in_cluster)

        # Store the title and sentences in the dictionary with the cluster number as the key
        clusters_with_titles[label] = title

    return clusters_with_titles
def get_clusters_as_string_with_titles(clusters):
    output_string = ""  # Initialize an empty string to store the output

    # Loop through each cluster and concatenate the sentences into the string
    for label, sentences_in_cluster in clusters.items():
        # Generate a title for the current cluster
        title = generate_topic_title(sentences_in_cluster)

        # Append the title and the topic number to the output string
        output_string += f"Topic {label + 1}: {title}\n"

        # Append each sentence to the output string
        for sentence in sentences_in_cluster:
            output_string += f" - {sentence}\n"
        output_string += "\n"  # Add a newline after each topic for better formatting

    return output_string
def get_clusters_with_titles(clusters):
    # Initialize an empty dictionary to store the output
    clusters_with_titles = {}

    # Loop through each cluster and generate the title
    for label, sentences_in_cluster in clusters.items():
        # Generate a title for the current cluster
        title = generate_topic_title(sentences_in_cluster)

        # Store the title and sentences in the dictionary with the cluster number as the key
        clusters_with_titles[label] = title

    return clusters_with_titles

def order_get_prominent_sentences_in_cluster(args, sentences, cluster_titles):
    # Sort the items in args by sentence index
    sorted_args = sorted(args.items(), key=lambda item: item[1])

    result = []

    for cluster_index, sentence_index in sorted_args:
        # Add the cluster title to the result
        title = cluster_titles[cluster_index]
        result.append(f"Cluster {cluster_index}: {title}")
        result.append(f"  {sentences[sentence_index]}")
        result.append("")  # Add a blank line for better readability

    return "\n".join(result)


def run_hdbscan_cluster(text, use_coreference_resolution=True, min_cluster_size=2, min_samples=2, cluster_selection_epsilon=0.8):
    text = clean_transcription(text)
    if use_coreference_resolution:
        text = coreference_resolution(text)
    sentences, embeddings = embed_sentences(text)
    cleaned_text = clean_transcription(text)
    cluster_labels = segment_topics_hdbscan(embeddings, min_cluster_size=min_cluster_size, min_samples=min_samples, cluster_selection_epsilon=cluster_selection_epsilon)
    clusters = putting_sentences_in_cluster(cluster_labels, sentences)
    min_sentences_for_subtopics = 7  # Threshold for creating subtopics
    max_subtopics = 3  # Maximum number of subtopics to generate

    subtopics_with_sentences = {}
    output_string = ""  # Initialize an empty string to store the output

    for topic, docs in clusters.items():
        output_string += f"Topic {topic}:\n"

        if len(docs) > min_sentences_for_subtopics:
            vectorizer = CountVectorizer(max_features=1000, stop_words='english')
            X = vectorizer.fit_transform(docs)

            lda = LatentDirichletAllocation(n_components=min(max_subtopics, len(docs) // 3), random_state=42)
            lda.fit(X)

            subtopic_labels = lda.transform(X).argmax(axis=1)
            subtopics_with_sentences[topic] = {}

            for subtopic in set(subtopic_labels):
                subtopic_sentences = [sentence for i, sentence in enumerate(docs) if subtopic_labels[i] == subtopic]
                keywords = [vectorizer.get_feature_names_out()[i] for i in lda.components_[subtopic].argsort()[-5:]]
                subtopic_name = " ".join(keywords)
                subtopics_with_sentences[topic][subtopic_name] = subtopic_sentences

                # Append the subtopic name and sentences to the output string
                output_string += f"  Subtopic: {subtopic_name}\n"
                for sentence in subtopic_sentences:
                    output_string += f"    - {sentence}\n"
                output_string += "\n"
        else:
            subtopics_with_sentences[topic] = docs

            # Append sentences directly if no subtopics are created
            for sentence in docs:
                output_string += f"  - {sentence}\n"
            output_string += "\n"

    return output_string


# ###################

# #helper functions for kmeans

def find_closest_args(centroids,embeddings) :
        centroid_min = 1e10
        cur_arg = -1
        args = {}
        used_idx = []

        for j, centroid in enumerate(centroids):

            for i, feature in enumerate(embeddings):
                value = np.linalg.norm(feature - centroid)

                if value < centroid_min and i not in used_idx:
                    cur_arg = i
                    centroid_min = value

            used_idx.append(cur_arg)
            args[j] = cur_arg
            centroid_min = 1e10
            cur_arg = -1

        return args

def calculate_elbow(embeddings,k_max,sentences_num):
    inertias = []
    for k in range(1,min(k_max + 1, int(sentences_num/2) + 1)):
        model = KMeans(n_clusters=k, random_state=42).fit(embeddings)
        inertias.append(model.inertia_)
    return inertias

def find_closest_sentences(centroids, embeddings):
    args = {}
    used_idx = []

    for j, centroid in enumerate(centroids):
        centroid_min = 1e10  # Reset minimum distance for each centroid
        cur_arg = -1

        for i, feature in enumerate(embeddings):
            value = np.linalg.norm(feature - centroid)  # Corrected line

            if value < centroid_min and i not in used_idx:
                cur_arg = i
                centroid_min = value

        used_idx.append(cur_arg)
        args[j] = cur_arg

    return args

def calculate_optimal_cluster(embeddings,k_max,sentences_num):
    delta_1 = []
    delta_2 = []

    max_strength = 0
    k = 1

    inertias = calculate_elbow(embeddings,k_max,sentences_num)

    for i in range(len(inertias)):
        delta_1.append(inertias[i] - inertias[i - 1] if i > 0 else 0.0)
        delta_2.append(delta_1[i] - delta_1[i - 1] if i > 1 else 0.0)

    for j in range(len(inertias)):
        strength = 0 if j <= 1 or j == len(inertias) - 1 else delta_2[j + 1] - delta_1[j + 1]

        if strength > max_strength:
            max_strength = strength
            k = j + 1

    return k

def applying_kmeans(embeddings,cluster_number=2):
  kmeans = KMeans(n_clusters=cluster_number, random_state=42)
  kmeans.fit(embeddings)
  return (kmeans.labels_ , kmeans.cluster_centers_)


def run_Kmeans_cluster(text,use_pca,n_components,use_coreference_resolution,cluster_number):
  text = clean_transcription(text)
  if use_coreference_resolution:
    text = coreference_resolution(text)
  sentences, embeddings = embed_sentences(text)
  if use_pca:
    if n_components >= len(sentences):
      n_components = int(len(sentences)/2)
    pca = PCA(n_components=n_components)
    embeddings = pca.fit_transform(embeddings)
  cluster_labels , cluster_centers = applying_kmeans(embeddings,cluster_number)
  return cluster_labels, cluster_centers,sentences

def run_predicted_clustering(text,n_components=40):
  text = clean_transcription(text)
  text = coreference_resolution(text)
  # run_Kmeans_cluster(text,True,40,True,cluster_number)
  sentences, embeddings = embed_sentences(text)
  if n_components >= len(sentences):
    n_components = int(len(sentences)/2)
  pca = PCA(n_components=n_components)
  embeddings = pca.fit_transform(embeddings)
  predicted_k=calculate_optimal_cluster(embeddings,len(sentences)//2,len(sentences)//2)
  cluster_labels , cluster_centers = applying_kmeans(embeddings,predicted_k)
  return cluster_labels, cluster_centers,sentences
###################################

#hdbscan
def run_hdbscan_cluster2(text, use_coreference_resolution=True, min_cluster_size=2, min_samples=2, cluster_selection_epsilon=0.8):
    text = clean_transcription(text)
    if use_coreference_resolution:
        text = coreference_resolution(text)
    sentences, embeddings = embed_sentences(text)
    cluster_labels = segment_topics_hdbscan(embeddings, min_cluster_size=min_cluster_size, min_samples=min_samples, cluster_selection_epsilon=cluster_selection_epsilon)
    clusters = putting_sentences_in_cluster(cluster_labels, sentences)
    min_sentences_for_subtopics = 7  # Threshold for creating subtopics
    max_subtopics = 3  # Maximum number of subtopics to generate

    subtopics_with_sentences = {}
    output_string = ""  # Initialize an empty string to store the output

    for topic, docs in clusters.items():
        # Skip outliers (topic == -1)
        if topic == -1:
            continue

        output_string += f"Topic {topic}:\n"

        if len(docs) > min_sentences_for_subtopics:
            vectorizer = CountVectorizer(max_features=1000, stop_words='english')
            X = vectorizer.fit_transform(docs)

            lda = LatentDirichletAllocation(n_components=min(max_subtopics, len(docs) // 3), random_state=42)
            lda.fit(X)

            subtopic_labels = lda.transform(X).argmax(axis=1)
            subtopics_with_sentences[topic] = {}

            for subtopic in set(subtopic_labels):
                subtopic_sentences = [sentence for i, sentence in enumerate(docs) if subtopic_labels[i] == subtopic]
                keywords = [vectorizer.get_feature_names_out()[i] for i in lda.components_[subtopic].argsort()[-5:]]
                subtopic_name = " ".join(keywords)
                subtopics_with_sentences[topic][subtopic_name] = subtopic_sentences

                # Append the subtopic name and sentences to the output string
                output_string += f"  Subtopic: {subtopic_name}\n"
                for sentence in subtopic_sentences:
                    output_string += f"    - {sentence}\n"
                output_string += "\n"
        else:
            subtopics_with_sentences[topic] = docs

            # Append sentences directly if no subtopics are created
            for sentence in docs:
                output_string += f"  - {sentence}\n"
            output_string += "\n"

    return output_string
###################################################

#kmeans
def generate_predicted_clustering_with_title(text):
  cluster_labels, cluster_centers,sentences = run_predicted_clustering(text)
  clusters = putting_sentences_in_cluster(cluster_labels,sentences)
  clusters_string = get_clusters_as_string_with_titles(clusters)
  return clusters_string
def generate_Kmeans_cluster(text,use_pca,n_components,use_coreference_resolution,cluster_number):
  cluster_labels, cluster_centers,sentences= run_Kmeans_cluster(text,use_pca,n_components,use_coreference_resolution,cluster_number)
  clusters = putting_sentences_in_cluster(cluster_labels,sentences)
  clusters_string = cluster_as_string(clusters)
  return clusters_string

def generate_predicted_clustering(text):
  cluster_labels, cluster_centers,sentences = run_predicted_clustering(text)
  clusters = putting_sentences_in_cluster(cluster_labels,sentences)
  clusters_string = cluster_as_string(clusters)
  return clusters_string

def generate_Kmeans_cluster_with_title(text,use_pca,n_components,use_coreference_resolution,cluster_number):
  cluster_labels, cluster_centers,sentences = run_Kmeans_cluster(text,use_pca,n_components,use_coreference_resolution,cluster_number)
  clusters = putting_sentences_in_cluster(cluster_labels,sentences)
  clusters_string = get_clusters_as_string_with_titles(clusters)
  return clusters_string

# def generate_predicted_clustering_with_title(text):
#   cluster_labels, cluster_centers,sentences = run_predicted_clustering(text)
#   clusters = putting_sentences_in_cluster(cluster_labels,sentences)
#   clusters_string = get_clusters_as_string_with_titles(clusters)
#   return clusters_string


def generate_summary(text,use_pca,n_components,use_coreference_resolution,cluster_number):
  # calculate_optimal_cluster(embeddings,k_max,sentences_num)
  text = clean_transcription(text)
  if use_coreference_resolution:
    text = coreference_resolution(text)
  sentences, embeddings = embed_sentences(text)
  if use_pca:
    if n_components >= len(sentences):
      n_components = int(len(sentences)/2)
    pca = PCA(n_components=n_components)
    embeddings = pca.fit_transform(embeddings)
  cluster_labels , cluster_centers = applying_kmeans(embeddings,cluster_number)
  clusters = putting_sentences_in_cluster(cluster_labels,sentences)
  linked_title_cluster=get_clusters_with_titles(clusters)
  args = find_closest_args(cluster_centers,embeddings)
  summary_text =  order_get_prominent_sentences_in_cluster(args, sentences,linked_title_cluster)
  return summary_text

def generate_summary2(text,use_pca,n_components,use_coreference_resolution,cluster_number):
  # calculate_optimal_cluster(embeddings,k_max,sentences_num)
  text = clean_transcription(text)
  if use_coreference_resolution:
    text = coreference_resolution(text)
  sentences, embeddings = embed_sentences(text)
  if use_pca:
    if n_components >= len(sentences):
      n_components = int(len(sentences)/2)
    pca = PCA(n_components=n_components)
    embeddings = pca.fit_transform(embeddings)
  cluster_labels , cluster_centers = applying_kmeans(embeddings,cluster_number)
  clusters = putting_sentences_in_cluster(cluster_labels,sentences)
  linked_title_cluster=get_clusters_with_titles(clusters)
  args = find_closest_args(cluster_centers,embeddings)
  summary_text =  order_get_prominent_sentences_in_cluster(args, sentences,linked_title_cluster)
  return summary_text
###################################################

#whisper
def formattedtime(seconds):
    final_time = time.strftime("%H:%M:%S", time.gmtime(float(seconds)))
    return f"{final_time},{seconds.split('.')[1]}"

def generate_transcription(file):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = WhisperModel("distil-large-v3", device=device)
    segments, _ = model.transcribe(file, beam_size=1)
    segments = list(segments)
    count = 0

    srt_text = ""  # Initialize an empty string to store the SRT content

    for segment in segments:
        count += 1
        start = formattedtime(format(segment.start, ".3f"))
        end = formattedtime(format(segment.end, ".3f"))
        txt = f"{count}\n{start} --> {end}\n{segment.text.strip()}\n\n"
        srt_text += txt  # Append the current subtitle to the string

    return srt_text  # Return the accumulated string

def run_hdbscan_cluster1(text, use_coreference_resolution=True, min_cluster_size=2, min_samples=2, cluster_selection_epsilon=0.8):
    text = clean_transcription(text)
    if use_coreference_resolution:
        text = coreference_resolution(text)
    sentences, embeddings = embed_sentences(text)
    cluster_labels = segment_topics_hdbscan(embeddings, min_cluster_size=min_cluster_size, min_samples=min_samples, cluster_selection_epsilon=cluster_selection_epsilon)
    clusters = putting_sentences_in_cluster(cluster_labels, sentences)
    min_sentences_for_subtopics = 7  # Threshold for creating subtopics
    max_subtopics = 3  # Maximum number of subtopics to generate

    subtopics_with_sentences = {}
    output_string = ""  # Initialize an empty string to store the output

    for topic, docs in clusters.items():
        if topic == -1:  # Handle outliers
            output_string += f"Outliers:\n"
            for sentence in docs:
                output_string += f"  - {sentence}\n"
            output_string += "\n"
            continue  # Skip to the next topic

        output_string += f"Topic {topic}:\n"

        if len(docs) > min_sentences_for_subtopics:
            vectorizer = CountVectorizer(max_features=1000, stop_words='english')
            X = vectorizer.fit_transform(docs)

            lda = LatentDirichletAllocation(n_components=min(max_subtopics, len(docs) // 3), random_state=42)
            lda.fit(X)

            subtopic_labels = lda.transform(X).argmax(axis=1)
            subtopics_with_sentences[topic] = {}

            for subtopic in set(subtopic_labels):
                subtopic_sentences = [sentence for i, sentence in enumerate(docs) if subtopic_labels[i] == subtopic]
                keywords = [vectorizer.get_feature_names_out()[i] for i in lda.components_[subtopic].argsort()[-5:]]
                subtopic_name = " ".join(keywords)
                subtopics_with_sentences[topic][subtopic_name] = subtopic_sentences

                # Append the subtopic name and sentences to the output string
                output_string += f"  Subtopic: {subtopic_name}\n"
                for sentence in subtopic_sentences:
                    output_string += f"    - {sentence}\n"
                output_string += "\n"
        else:
            subtopics_with_sentences[topic] = docs

            # Append sentences directly if no subtopics are created
            for sentence in docs:
                output_string += f"  - {sentence}\n"
            output_string += "\n"

    return output_string


#main functions

# def complete_hdbscan(srt_text):
#   string = run_hdbscan_cluster2(srt_text)
#   return string

def complete_hdbscan(srt_text):
  string =run_hdbscan_cluster(srt_text)
  return string

def complete_predicted_kmeans(srt_text):
  string = generate_predicted_clustering_with_title(srt_text)
  return string

def complete_kmeans(srt_text,cluster_number):
  string =generate_Kmeans_cluster_with_title(srt_text,True,40,True,cluster_number)
  return string

def complete_summary(srt_text,cluster_number):
  string =generate_summary(srt_text,True,40,True,cluster_number)
  return string


def complete_summary1(srt_text,cluster_number):
  string =generate_summary2(srt_text,True,40,True,cluster_number)
  return string




app = FastAPI()

# # This defines the data json format expected for the endpoint, change as needed
# class TextInput(BaseModel):
#     inputs: str
#     parameters: dict[str, Any] | None

@app.get("/")
def status_gpu_check() :
    # gpu_msg = "Available" if tf.test.is_gpu_available() else "Unavailable"
    return {
        "status": "I am ALIVE!",
        "gpu": "gpu_msg"
    }

@app.post("/generate/")
async def generate_text() :
    try:
        print("type(data)")
        print("data")
        # params = data.parameters or {}
        # response = llama2_model(prompt=data.inputs, **params)
        # model_out = response['choices'][0]['text']
        return {"generated_text": "model_out"}
    except Exception as e:
        print("type(data)failed")
        print("data failed")
        raise HTTPException(status_code=500, detail=len(str(e)))

@app.post("/file/hdbscan/")
async def file_hdbscan(file: UploadFile = File(...)):
    try:
        file_content = await file.read()
        decoded_content = file_content.decode('utf-8')
        hdbscan_string = complete_hdbscan(decoded_content)
        return {"generated_text": hdbscan_string}

    except Exception as e:
        # Error handling
        raise HTTPException(status_code=500, detail=f"An error occurred: {str(e)}")

@app.post("/video/hdbscan/")
async def video_hdbscan(file: UploadFile = File(...), param1: int = Form(...)):
    try:
        # Create a temporary directory to save the video file
        temp_dir = tempfile.mkdtemp()
        temp_file_path = os.path.join(temp_dir, file.filename)

        # Save the uploaded video file to the temporary directory
        with open(temp_file_path, "wb") as temp_file:
            shutil.copyfileobj(file.file, temp_file)

        # Process the video (You can add your video processing code here)
        # For example, let's assume we just print the file path
        # Simulate some processing (e.g., analyzing or converting the video)
        # You can add your processing logic here
        srt_content = generate_transcription(temp_file_path)
        hdbscan_string = complete_hdbscan(srt_content)
        return {"generated_text": hdbscan_string}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"An error occurred: {str(e)}")

    finally:
        # Make sure to delete the temporary file and directory after processing
        try:
            os.remove(temp_file_path)  # Delete the temporary file
            os.rmdir(temp_dir)  # Remove the temporary directory
        except Exception as cleanup_error:
            print(f"Error during cleanup: {cleanup_error}")


@app.post("/file/kmeans/predicted/")
async def file_kmeans_predicted(file: UploadFile = File(...)):
    try:
        file_content = await file.read()
        decoded_content = file_content.decode('utf-8')
        predicted_kmeans_string = complete_predicted_kmeans(decoded_content)
        return {"generated_text": predicted_kmeans_string}

    except Exception as e:
        # Error handling
        raise HTTPException(status_code=500, detail=f"An error occurred: {str(e)}")


@app.post("/file/kmeans/")
async def file_kmeans(file: UploadFile = File(...), param1: int = Form(...)):
    try:
        file_content = await file.read()
        decoded_content = file_content.decode('utf-8')
        kmeans_string = complete_kmeans(decoded_content,param1)
        return {"generated_text": kmeans_string}

    except Exception as e:
        # Error handling
        raise HTTPException(status_code=500, detail=f"An error occurred: {str(e)}")

@app.post("/video/kmeans/")
async def video_kmeans(file: UploadFile = File(...), param1: int = Form(...)):
    try:
        # Create a temporary directory to save the video file
        temp_dir = tempfile.mkdtemp()
        temp_file_path = os.path.join(temp_dir, file.filename)

        # Save the uploaded video file to the temporary directory
        with open(temp_file_path, "wb") as temp_file:
            shutil.copyfileobj(file.file, temp_file)

        # Process the video (You can add your video processing code here)
        # For example, let's assume we just print the file path
        # Simulate some processing (e.g., analyzing or converting the video)
        # You can add your processing logic here
        srt_content = generate_transcription(temp_file_path)
        kmeans_string = complete_kmeans(srt_content,param1)
        return {"generated_text": kmeans_string}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"An error occurred: {str(e)}")

    finally:
        # Make sure to delete the temporary file and directory after processing
        try:
            os.remove(temp_file_path)  # Delete the temporary file
            os.rmdir(temp_dir)  # Remove the temporary directory
        except Exception as cleanup_error:
            print(f"Error during cleanup: {cleanup_error}")

@app.post("/file/summary/")
async def fille_summary(file: UploadFile = File(...), param1: int = Form(...)):
    try:
        file_content = await file.read()
        decoded_content = file_content.decode('utf-8')
        summary_string = complete_summary1(decoded_content,param1)
        return {"generated_text": summary_string}

    except Exception as e:
        # Error handling
        raise HTTPException(status_code=500, detail=f"An error occurred: {str(e)}")

@app.post("/video/summary/")
async def video_summary(file: UploadFile = File(...), param1: int = Form(...)):
    try:
        # Create a temporary directory to save the video file
        temp_dir = tempfile.mkdtemp()
        temp_file_path = os.path.join(temp_dir, file.filename)

        # Save the uploaded video file to the temporary directory
        with open(temp_file_path, "wb") as temp_file:
            shutil.copyfileobj(file.file, temp_file)

        # Process the video (You can add your video processing code here)
        # For example, let's assume we just print the file path
        # Simulate some processing (e.g., analyzing or converting the video)
        # You can add your processing logic here
        srt_content = generate_transcription(temp_file_path)
        summary_string = complete_summary1(srt_content,param1)
        return {"generated_text": summary_string}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"An error occurred: {str(e)}")

    finally:
        # Make sure to delete the temporary file and directory after processing
        try:
            os.remove(temp_file_path)  # Delete the temporary file
            os.rmdir(temp_dir)  # Remove the temporary directory
        except Exception as cleanup_error:
            print(f"Error during cleanup: {cleanup_error}")

# @app.post("/video")
# async def video(file: UploadFile = File(...), param1: int = Form(...)):
#     try:
#         file_content = await file.read()
#         decoded_content = file_content.decode('utf-8')
#         summary_string = complete_summary(decoded_content,param1)
#         return {"generated_text": summary_string}

#     except Exception as e:
#         # Error handling
#         raise HTTPException(status_code=500, detail=f"An error occurred: {str(e)}")






@app.post("/video/kmeans/predicted/")
async def video_predicted_kmeans(file: UploadFile = File(...)):
    try:
        # Create a temporary directory to save the video file
        temp_dir = tempfile.mkdtemp()
        temp_file_path = os.path.join(temp_dir, file.filename)

        # Save the uploaded video file to the temporary directory
        with open(temp_file_path, "wb") as temp_file:
            shutil.copyfileobj(file.file, temp_file)

        # Process the video (You can add your video processing code here)
        # For example, let's assume we just print the file path
        print(f"Processing video: {temp_file_path}")

        # Simulate some processing (e.g., analyzing or converting the video)
        # You can add your processing logic here
        srt_content = generate_transcription(temp_file_path)
        predicted_kmeans_string = complete_predicted_kmeans(srt_content)
        return {"generated_text": predicted_kmeans_string}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"An error occurred: {str(e)}")

    finally:
        # Make sure to delete the temporary file and directory after processing
        try:
            os.remove(temp_file_path)  # Delete the temporary file
            os.rmdir(temp_dir)  # Remove the temporary directory
        except Exception as cleanup_error:
            print(f"Error during cleanup: {cleanup_error}")


Overwriting app.py


In [ ]:
def read_file(path):
  with open(path, 'r') as file:
    file_content = file.read()
  return file_content

In [ ]:
read_file()

In [ ]:
!pip install -q -U google-generativeai

In [ ]:
from google.colab import files

# Upload file
uploaded = files.upload()

Saving 04_week-introduction.mp4 to 04_week-introduction.mp4


In [ ]:
!ls

04_week-introduction.mp4  app.py  sample_data  server.log


In [ ]:
# The server will start the model download and will take a while to start up
# ~5 minutes if its not already downloaded

import subprocess
import time

from ipywidgets import HTML
from IPython.display import display

t = HTML(
    value="0 Seconds",
    description = 'Server is Starting Up... Elapsed Time:' ,
    style={'description_width': 'initial'},
)
display(t)

flag = True
timer = 0

try:
    subprocess.check_output(['curl',"localhost:8000"])
    flag = False
except:
    get_ipython().system_raw('uvicorn app:app --host 0.0.0.0 --port 8000 > server.log 2>&1 &')

res = ""

while(flag and timer < 600):
  try:
    subprocess.check_output(['curl',"localhost:8000"])
  except:
    time.sleep(1)
    timer+= 1
    t.value = str(timer) + " Seconds"
    pass
  else:
    flag = False

if(timer >= 600):
  print("Error: timed out! took more then 10 minutes :(")
subprocess.check_output(['curl',"localhost:8000"])

HTML(value='0 Seconds', description='Server is Starting Up... Elapsed Time:', style=DescriptionStyle(descripti…

b'{"status":"I am ALIVE!","gpu":"gpu_msg"}'

In [ ]:
# This starts Ngrok and creates the public URL
import subprocess
import time
import sys
import json

from IPython import get_ipython
get_ipython().system_raw('ngrok http 8000 &')
time.sleep(1)
curlOut = subprocess.check_output(['curl',"http://localhost:4040/api/tunnels"],universal_newlines=True)
time.sleep(1)
ngrokURL = json.loads(curlOut)['tunnels'][0]['public_url']
%store ngrokURL
print(ngrokURL)

CalledProcessError: Command '['curl', 'http://localhost:4040/api/tunnels']' returned non-zero exit status 7.

In [ ]:
!ls

04_week-introduction.mp4  app.py  __pycache__  sample_data  server.log


In [ ]:
import torch
import io
import soundfile as sf
from faster_whisper import WhisperModel

In [ ]:
def formattedtime(time):
    hours = int(time // 3600)
    minutes = int((time % 3600) // 60)
    seconds = int(time % 60)
    milliseconds = int((time - int(time)) * 1000)
    return f"{hours:02}:{minutes:02}:{seconds:02},{milliseconds:03}"

def generate_transcription(audio_bytes):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = WhisperModel("distil-large-v3", device=device)

    # Convert bytes to an in-memory file object
    audio_file = io.BytesIO(audio_bytes)

    # Read the audio data from the file object
    audio_data, sample_rate = sf.read(audio_file)

    # Ensure audio is in the correct format (convert to mono if needed)
    if len(audio_data.shape) > 1:  # Stereo to Mono
        audio_data = audio_data.mean(axis=1)

    # Transcribe the audio data
    segments, _ = model.transcribe(audio_data, beam_size=1)

    srt_text = ""  # Initialize an empty string to store the SRT content
    count = 0

    for segment in segments:
        count += 1
        start = formattedtime(float(format(segment.start, ".3f")))
        end = formattedtime(float(format(segment.end, ".3f")))
        txt = f"{count}\n{start} --> {end}\n{segment.text.strip()}\n\n"
        srt_text += txt  # Append the current subtitle to the string

    return srt_text  # Return the accumulated string

In [ ]:
import requests
# Define the URL for the FastAPI endpoint
%store -r ngrokURL

# Define the data to send in the POST request
# data = {
#   "inputs": '''
# Tell me how to make a chocolate cake?
# ''',
#   #paramaters can be found here https://abetlen.github.io/llama-cpp-python/#llama_cpp.llama.Llama.create_completion
#   "parameters": {"temperature":0.1,
#                  "max_tokens":200}
#   #higher temperature, more creative response is, lower more precise
#   #max_token is the max amount of (simplified) "words" allowed to be generated
# }
file_path = "04_week-introduction.mp4"  # Replace with the actual path to your file

# Open the file in binary mode
# with open(file_path, "rb") as file:
#     response = requests.post(ngrokURL + "/generate/", files={"file": file})

# with open(file_path, "rb") as file:
#     response = requests.post(ngrokURL + "/file/kmeans/predicted/", files={"file": file})

# with open(file_path, "rb") as file:
#     response = requests.post(ngrokURL + "/video/kmeans/predicted/", files={"file": file})

# with open(file_path, "rb") as file:
#     params = {'param1': 3}
#     response = requests.post(ngrokURL + "/file/kmeans/", files={"file": file}, data=params)

# with open(file_path, "rb") as file:
#     params = {'param1': 3}
#     response = requests.post(ngrokURL + "/file/summary/", files={"file": file}, data=params)

with open(file_path, "rb") as file:
    response = requests.post(ngrokURL + "/file/hdbscan/", files={"file": file})

# with open(file_path, "rb") as file:
#     params = {'param1': 42, 'param2': 24}
#     response = requests.post(ngrokURL + "/file/kmeans/predicted/", files={"file": file}, data=params)

# # Send the POST request
# response = requests.post(ngrokURL + "/generate/", json=data)
# /video/kmeans/
# with open(file_path, "rb") as file:
#     response = requests.post(ngrokURL + "/video/hdbscan/", files={"file": file})

# جرب هدول يسر
# with open(file_path, "rb") as file:
#     params = {'param1': 3}
#     response = requests.post(ngrokURL + "/video/kmeans/", files={"file": file}, data=params)
# with open(file_path, "rb") as file:
#     params = {'param1': 3}
#     response = requests.post(ngrokURL + "/video/summary/", files={"file": file}, data=params)
# with open(file_path, "rb") as file:
#     response = requests.post(ngrokURL + "/video/kmeans/predicted/", files={"file": file}, data=params)
# with open(file_path, "rb") as file:
#     response = requests.post(ngrokURL + "/video/hdbscan/", files={"file": file})

if response.status_code == 200:
    result = response.json()
    print("Generated Text:\n", result)
else:
    try:
        error_info = response.json()
    except ValueError:
        error_info = response.text
    print("Request failed with status code:", response.status_code)
    print("Error details:", error_info)
# Check the response
# if response.status_code == 200:
#     result = response.json()
#     # print("Generated Text:\n", data["inputs"], result["generated_text"].strip())
#     print("Generated Text:\n", result)
# else:
#     print("Request failed with status code:", response.status_code)

In [ ]:
!pkill uvicorn

In [ ]:
!pkill ngrok

In [ ]:
import shutil

In [ ]:
file_path = "04_week-introduction.mp4"

In [ ]:
with file_path.open("wb") as buffer:
  shutil.copyfileobj(file.file, buffer)

AttributeError: 'str' object has no attribute 'open'

# Helper Funtions


In [ ]:
p = result["generated_text"]
print(p)

Topic 1: ll logistic week
 - Welcome to the first week of Course 1.
the first week of Course 1 is all about logistic regression, which is a very important tool used in many applications
in NLP.

 - Logistic regression algorithms are particularly useful because Logistic regression algorithms are easy to train and
provide you with a good baseline result.

 - the first week of Course 1 you'll use logistic regression for sentiment analysis of tweets.

 - You'll first process your data, then you train your model and finally you'll test the accuracy
of your model.

 - We've designed these courses so that each week Jonas'll give you an overview of the material and
And Jonas will give you the details.

 - So take it away, Jonas.

 - Okay, great.

 - Follow Jonas.


